# 2022111723 데이터사이언스학과 김경은

In [ ]:
!pip install torch torchvision opencv-python matplotlib segment-anything
!pip install timm einops
!pip install gradio
!pip install -qq -U diffusers==0.11.1 transformers ftfy gradio accelerate
!pip install --upgrade diffusers transformers safetensors accelerate

### 1. 시각화를 위한 함수 정의

In [ ]:
import numpy as np
import torch
import matplotlib.pyplot as plt
import cv2

# 마스크를 시각화하는 함수
def show_mask(mask, ax, random_color=False):
    if random_color:
        # 임의의 색상 생성
        color = np.concatenate([np.random.random(3), np.array([0.6])], axis=0)
    else:
        # 고정된 색상 (파란색, 투명도 0.6)
        color = np.array([30/255, 144/255, 255/255, 0.6])
    h, w = mask.shape[-2:] # 마스크의 높이와 너비 가져오기
    # 마스크를 색상과 결합하여 이미지 생성
    mask_image = mask.reshape(h, w, 1) * color.reshape(1, 1, -1)
    # Axes 객체에 마스크 이미지 표시
    ax.imshow(mask_image)

# 좌표 및 레이블을 시각화하는 함수
def show_points(coords, labels, ax, marker_size=375):
    # Positive 레이블 포인트 가져오기
    pos_points = coords[labels==1]
    # Negative 레이블 포인트 가져오기
    neg_points = coords[labels==0]
    # Positive 포인트를 초록색 별로 표시
    ax.scatter(pos_points[:, 0], pos_points[:, 1], color='green', marker='*', s=marker_size, edgecolor='white', linewidth=1.25)
    # Negative 포인트를 빨간색 별로 표시
    ax.scatter(neg_points[:, 0], neg_points[:, 1], color='red', marker='*', s=marker_size, edgecolor='white', linewidth=1.25)

### 2. Segment Anything Model (SAM)

In [ ]:
using_colab = True

In [ ]:
if using_colab:
    import torch
    import torchvision
    print("PyTorch version:", torch.__version__)
    print("Torchvision version:", torchvision.__version__)
    print("CUDA is available:", torch.cuda.is_available())
    import sys
    !{sys.executable} -m pip install opencv-python matplotlib
    !{sys.executable} -m pip install 'git+https://github.com/facebookresearch/segment-anything.git'

    !wget https://dl.fbaipublicfiles.com/segment_anything/sam_vit_h_4b8939.pth

In [ ]:
import sys
sys.path.append("..")
from segment_anything import sam_model_registry, SamPredictor

sam_checkpoint = "sam_vit_h_4b8939.pth"
model_type = "vit_h"

device = "cuda"

sam = sam_model_registry[model_type](checkpoint=sam_checkpoint)
sam.to(device=device)
predictor = SamPredictor(sam)

In [ ]:
# 이미지 로드 및 전처리
image_path = "도넛.png"
image = cv2.imread(image_path)
image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
predictor.set_image(image)

In [ ]:
plt.figure(figsize=(10,10))
plt.imshow(image)
plt.axis('on')
plt.show()

In [ ]:
# 포인트 및 라벨 설정
input_point = np.array([[170, 250]]) # 좌표값 입력
input_label = np.array([1])

In [ ]:
plt.figure(figsize=(10,10))
plt.imshow(image)
show_points(input_point, input_label, plt.gca())
plt.axis('on')
plt.show()

In [ ]:
# SAM으로 마스크 예측
masks, scores, logits = predictor.predict(
    point_coords=input_point,
    point_labels=input_label,
    multimask_output=True,
)

In [ ]:
# 반환된 masks의 배열 크기를 확인
masks.shape  # (number_of_masks) x H x W

In [ ]:
# 가장 높은 점수를 가진 마스크 선택
best_mask_idx = np.argmax(scores)
best_mask = masks[best_mask_idx]

In [ ]:
# 마스크 확장
kernel_size = 30  # 확장 범위 설정
kernel = np.ones((kernel_size, kernel_size), np.uint8)
dilated_mask = cv2.dilate(best_mask.astype(np.uint8), kernel, iterations=1)

In [ ]:
# 모든 마스크 시각화
for i, (mask, score) in enumerate(zip(masks, scores)):
    plt.figure(figsize=(10, 10))
    plt.imshow(image)
    show_mask(mask, plt.gca())  # 기본 마스크 표시
    show_points(input_point, input_label, plt.gca())
    plt.title(f"Mask {i+1}, Score: {score:.3f}", fontsize=18)
    plt.axis('off')
    plt.show()


In [ ]:
# 최고 점수 마스크에 확장 적용 시각화
plt.figure(figsize=(10, 10))
plt.imshow(image)
show_mask(dilated_mask, plt.gca())  # 확장된 마스크 표시
show_points(input_point, input_label, plt.gca())
plt.title(f"Expanded Mask (Best Score: {scores[best_mask_idx]:.3f})", fontsize=18)
plt.axis('off')
plt.show()

#### 3. Stable Diffusion Inpainting

In [ ]:
# 필요한 라이브러리 호출
import numpy as np
import torch
import cv2
import matplotlib.pyplot as plt
from segment_anything import sam_model_registry, SamPredictor
from diffusers import StableDiffusionInpaintPipeline
import PIL
from PIL import Image

In [ ]:
# 마스크 이미지로 변환
mask_image = (best_mask * 255).astype(np.uint8)
mask_image = Image.fromarray(mask_image).convert("L")
mask_image = mask_image.resize((512, 512))

In [ ]:
mask_image

In [ ]:
# mask_image 저장
mask_image.save("mask_image.png")
print("Mask image saved as mask_image.png")

In [ ]:
# 이미지 리사이즈 및 PIL 변환
image_resized = Image.fromarray(image).resize((512, 512))

In [ ]:
# Inpainting 모델 설정
model_path = "runwayml/stable-diffusion-inpainting"
pipe = StableDiffusionInpaintPipeline.from_pretrained(
    model_path,
    torch_dtype=torch.float16,
).to(device)

In [ ]:
# Inpainting 프롬프트
prompt = "Chocolate donuts topped with sprinkles."
guidance_scale = 7.5
num_samples = 3
generator = torch.Generator(device=device).manual_seed(42)  # 동일한 결과를 위해 고정 시드 사용

In [ ]:
# Inpainting 실행
images = pipe(
    prompt=prompt,
    image=image_resized,
    mask_image=mask_image,
    guidance_scale=guidance_scale,
    generator=generator,
    num_images_per_prompt=num_samples,
).images

In [ ]:
# 초기 이미지를 포함하여 비교
images.insert(0, image_resized)

In [ ]:
# 결과 표시
def image_grid(imgs, rows, cols):
    assert len(imgs) == rows * cols
    w, h = imgs[0].size
    grid = PIL.Image.new("RGB", size=(cols * w, rows * h))
    for i, img in enumerate(imgs):
        grid.paste(img, box=(i % cols * w, i // cols * h))
    return grid

In [ ]:
image_grid(images, 1, num_samples + 1)

## Gradio를 통한 Stable Diffusion Inpainting 수행

In [ ]:
import gradio as gr
import torch
from PIL import Image
from diffusers import StableDiffusionInpaintPipeline

# 모델 설정
model_path = "runwayml/stable-diffusion-inpainting"
device = "cuda" if torch.cuda.is_available() else "cpu"

pipe = StableDiffusionInpaintPipeline.from_pretrained(
    model_path,
    torch_dtype=torch.float16
).to(device)

# Remove safety checker
pipe.safety_checker = None

# 이미지 그리드 생성 함수
def image_grid(imgs, rows, cols):
    assert len(imgs) == rows * cols
    w, h = imgs[0].size
    grid = Image.new("RGB", size=(cols * w, rows * h))
    for i, img in enumerate(imgs):
        grid.paste(img, box=(i % cols * w, i // cols * h))
    return grid

# Inpainting 함수
def run_inpainting(image, mask_image, prompt, guidance_scale, num_samples):
    if image is None or mask_image is None:
        return "Please upload both an image and a mask."

    # 리사이즈
    image_resized = image.resize((512, 512))
    mask_resized = mask_image.resize((512, 512))

    # Inpainting 실행
    generator = torch.Generator(device=device).manual_seed(42)
    images = pipe(
        prompt=prompt,
        image=image_resized,
        mask_image=mask_resized,
        guidance_scale=guidance_scale,
        generator=generator,
        num_images_per_prompt=num_samples,
    ).images

    # 초기 이미지 포함하여 결과 생성
    images.insert(0, image_resized)
    return image_grid(images, rows=1, cols=num_samples + 1)

# Gradio 인터페이스
with gr.Blocks(css=".gradio-container { font-family: 'Arial', sans-serif; }") as demo:
    gr.Markdown(
        """
        <h1 style="text-align: center; color: #4CAF50;">Stable Diffusion Inpainting</h1>
        <p style="text-align: center;">Upload an image, provide a mask, and enter a prompt to generate inpainted images.</p>
        """
    )

    with gr.Row():
        with gr.Column(scale=1, min_width=300):
            input_image = gr.Image(label="Upload Image", type="pil", elem_id="input-image")
            mask_image = gr.Image(label="Upload Mask Image", type="pil", elem_id="mask-image")
            prompt = gr.Textbox(label="Inpainting Prompt", placeholder="Enter your prompt here...", lines=2)
            guidance_scale = gr.Slider(1.0, 10.0, value=7.5, step=0.1, label="Guidance Scale")
            num_samples = gr.Slider(1, 5, value=3, step=1, label="Number of Samples")
            run_button = gr.Button("Run Inpainting", elem_id="run-button")

        with gr.Column(scale=2):
            output_image = gr.Image(label="Generated Images", interactive=False, elem_id="output-image")

    run_button.click(
        fn=run_inpainting,
        inputs=[input_image, mask_image, prompt, guidance_scale, num_samples],
        outputs=output_image
    )

    gr.Markdown(
        """
        <style>
        #input-image, #mask-image, #output-image {
            border: 1px solid #ddd;
            box-shadow: 2px 2px 5px rgba(0, 0, 0, 0.1);
            border-radius: 5px;
        }
        #run-button {
            background-color: #4CAF50;
            color: white;
            padding: 10px 20px;
            font-size: 16px;
            border: none;
            cursor: pointer;
            border-radius: 5px;
        }
        #run-button:hover {
            background-color: #45a049;
        }
        </style>
        """
    )

# Launch the Gradio app
demo.launch(debug=True)